# Fidelity Metrics for XAI Validation: Deletion and Insertion Tests

## Overview

This notebook implements quantitative fidelity metrics to validate LIME explanations for the Diabetic Retinopathy classification model. The Deletion and Insertion tests measure how "faithful" an explanation is to the model's actual inner logic.

### Key Concept
If LIME says a specific cluster of pixels is the reason for a "Severe DR" prediction, then removing those pixels should cause the model's confidence to crash. This provides mathematical proof that the LIME-identified regions are genuinely the causal drivers of the model's predictions.

### Metrics
1. **Deletion AUC**: Remove pixels in order of importance (most to least). Lower AUC = more faithful explanation.
2. **Insertion AUC**: Add pixels in order of importance to a blank image. Higher AUC = more faithful explanation.

### Comparison
We compare LIME-ordered removal/insertion against a random baseline to quantify the improvement in explanation faithfulness.

In [ ]:
# ==============================================================================
# SECTION 1: IMPORTS AND SETUP
# ==============================================================================
import os
import json
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from torchvision import models, transforms
from skimage.segmentation import slic
from skimage.util import img_as_float
import random
import warnings
warnings.filterwarnings('ignore')

# LIME imports
from lime.lime_image import LimeImageExplainer

# --- Device Setup ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")

In [4]:
import json

In [ ]:
import json
import torch
import torch.nn as nn
from torchvision import models

In [ ]:
import torch
device = torch.device("cpu")
model = models.swin_t(weights=None).to(device)

In [1]:
print("step 1")

step 1


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

In [2]:
model = models.swin_t(weights=None)
print("step 2")

NameError: name 'models' is not defined

In [5]:
# ==============================================================================
# SECTION 2: LOAD MODEL AND CONFIGURATION
# ==============================================================================

# --- Load Config ---
with open('dataset_config.json', 'r') as f:
    config = json.load(f)
NUM_CLASSES = config['num_classes']
CLASS_MAPPING = config['class_to_idx']
idx_to_class = {v: k for k, v in CLASS_MAPPING.items()}
class_names = [idx_to_class[i] for i in range(NUM_CLASSES)]
print(f"✓ Loaded configuration for {NUM_CLASSES} classes: {class_names}")

# --- Load the Model ---
model = models.swin_t(weights=None)
in_features = model.head.in_features
model.head = nn.Linear(in_features, NUM_CLASSES)

model_weights_path = '../models/swin_transformer_final_generalized.pth'
model.load_state_dict(torch.load(model_weights_path, map_location=device))
model.eval()
model = model.to(device)
print(f"✓ Trained model loaded from: {model_weights_path}")

✓ Loaded configuration for 5 classes: ['No_DR', 'Mild', 'Moderate', 'Severe', 'Proliferative_DR']


NameError: name 'models' is not defined

In [ ]:
# ==============================================================================
# SECTION 3: DEFINE TRANSFORMS AND HELPER FUNCTIONS
# ==============================================================================

# Standard transforms for model input
def get_transform():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def preprocess_image(image_path):
    """Load and preprocess an image for model inference."""
    image = Image.open(image_path).convert('RGB')
    original_size = image.size  # (width, height)
    transform = get_transform()
    image_tensor = transform(image).unsqueeze(0).to(device)
    return image, image_tensor, original_size

def get_model_prediction(image_tensor):
    """Get model prediction and confidence score."""
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, predicted = torch.max(probabilities, 1)
    return predicted.item(), confidence.item(), probabilities.cpu().numpy()[0]

In [ ]:
# ==============================================================================
# SECTION 4: LIME EXPLANATION GENERATOR
# ==============================================================================

def generate_lime_explanation(image_pil, num_samples=1000, num_features=10):
    """
    Generate LIME explanation for an image.
    
    Args:
        image_pil: PIL Image object
        num_samples: Number of perturbed samples for LIME
        num_features: Maximum number of features to consider
    
    Returns:
        segments: Superpixel segmentation
        importance_scores: Dictionary mapping segment_id to importance score
        predicted_class: Model's predicted class
    """
    # Convert PIL to numpy array
    image_np = np.array(image_pil.resize((224, 224)))
    
    # Create LIME explainer
    explainer = LimeImageExplainer()
    
    # Define prediction function for LIME
    def predict_fn(images):
        """Batch prediction function for LIME."""
        transform = get_transform()
        batch_tensors = []
        for img in images:
            # Convert numpy array to PIL, then to tensor
            pil_img = Image.fromarray(img.astype('uint8'))
            tensor = transform(pil_img)
            batch_tensors.append(tensor)
        batch = torch.stack(batch_tensors).to(device)
        
        with torch.no_grad():
            outputs = model(batch)
            probs = torch.softmax(outputs, dim=1)
        return probs.cpu().numpy()
    
    # Generate explanation
    explanation = explainer.explain_instance(
        image_np,
        classifier_fn=predict_fn,
        top_labels=1,
        hide_color=0,
        num_samples=num_samples
    )
    
    predicted_class = explanation.top_labels[0]
    
    # Get segmentation
    segments = explanation.segments
    
    # Get local prediction weights (importance scores)
    local_pred = explanation.local_pred[predicted_class]
    
    # Map feature IDs to importance scores
    importance_scores = dict(explanation.local_exp[predicted_class])
    
    return segments, importance_scores, predicted_class

In [ ]:
# ==============================================================================
# SECTION 5: FIDELITY METRICS - DELETION TEST
# ==============================================================================

def deletion_test(image_pil, segments, importance_scores, predicted_class, 
                  num_steps=20, replacement_value=0):
    """
    Perform Deletion Test: Remove pixels in order of importance and track model confidence.
    
    The test progressively removes (replaces with a baseline value) the most important
    superpixels identified by LIME. If LIME correctly identifies important regions,
    the model's confidence should drop rapidly.
    
    Args:
        image_pil: Original PIL Image
        segments: Superpixel segmentation map (H x W)
        importance_scores: Dictionary of {segment_id: importance}
        predicted_class: Target class to track confidence for
        num_steps: Number of deletion steps
        replacement_value: Value to replace deleted pixels (0=black, 128=gray)
    
    Returns:
        deletion_scores: List of confidence scores at each step
        deletion_auc: Area under the deletion curve
    """
    # Prepare image
    image_np = np.array(image_pil.resize((224, 224))).astype('float32')
    
    # Sort segments by importance (most important first for LIME)
    sorted_segments = sorted(importance_scores.items(), key=lambda x: abs(x[1]), reverse=True)
    segment_ids_ordered = [s[0] for s in sorted_segments]
    
    total_segments = len(segment_ids_ordered)
    segments_per_step = max(1, total_segments // num_steps)
    
    deletion_scores = []
    modified_image = image_np.copy()
    
    # Get initial confidence
    transform = get_transform()
    pil_img = Image.fromarray(modified_image.astype('uint8'))
    tensor = transform(pil_img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(tensor)
        probs = torch.softmax(output, dim=1)
        initial_confidence = probs[0, predicted_class].item()
    
    deletion_scores.append(initial_confidence)
    
    # Progressive deletion
    deleted_segments = set()
    
    for step in range(num_steps):
        # Determine which segments to delete in this step
        start_idx = step * segments_per_step
        end_idx = min(start_idx + segments_per_step, total_segments)
        
        for idx in range(start_idx, end_idx):
            seg_id = segment_ids_ordered[idx]
            deleted_segments.add(seg_id)
            # Replace pixels in this segment with baseline
            mask = segments == seg_id
            modified_image[mask] = replacement_value
        
        # Get model confidence after deletion
        pil_img = Image.fromarray(modified_image.astype('uint8'))
        tensor = transform(pil_img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(tensor)
            probs = torch.softmax(output, dim=1)
            confidence = probs[0, predicted_class].item()
        
        deletion_scores.append(confidence)
        
        if end_idx >= total_segments:
            break
    
    # Calculate AUC (lower is better for deletion)
    x = np.linspace(0, 1, len(deletion_scores))
    deletion_auc = np.trapz(deletion_scores, x)
    
    return deletion_scores, deletion_auc

In [ ]:
# ==============================================================================
# SECTION 6: FIDELITY METRICS - INSERTION TEST
# ==============================================================================

def insertion_test(image_pil, segments, importance_scores, predicted_class,
                   num_steps=20, replacement_value=128):
    """
    Perform Insertion Test: Add pixels in order of importance and track model confidence.
    
    The test starts with a blank/blurred image and progressively inserts the most
    important superpixels identified by LIME. If LIME correctly identifies important
    regions, the model's confidence should rise rapidly.
    
    Args:
        image_pil: Original PIL Image
        segments: Superpixel segmentation map (H x W)
        importance_scores: Dictionary of {segment_id: importance}
        predicted_class: Target class to track confidence for
        num_steps: Number of insertion steps
        replacement_value: Value for non-inserted regions (128=gray, 0=black)
    
    Returns:
        insertion_scores: List of confidence scores at each step
        insertion_auc: Area under the insertion curve
    """
    # Prepare image
    image_np = np.array(image_pil.resize((224, 224))).astype('float32')
    
    # Sort segments by importance (most important first)
    sorted_segments = sorted(importance_scores.items(), key=lambda x: abs(x[1]), reverse=True)
    segment_ids_ordered = [s[0] for s in sorted_segments]
    
    total_segments = len(segment_ids_ordered)
    segments_per_step = max(1, total_segments // num_steps)
    
    insertion_scores = []
    
    # Start with blank image (all gray or black)
    modified_image = np.ones_like(image_np) * replacement_value
    
    # Get initial confidence on blank image
    transform = get_transform()
    pil_img = Image.fromarray(modified_image.astype('uint8'))
    tensor = transform(pil_img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(tensor)
        probs = torch.softmax(output, dim=1)
        initial_confidence = probs[0, predicted_class].item()
    
    insertion_scores.append(initial_confidence)
    
    # Progressive insertion
    inserted_segments = set()
    
    for step in range(num_steps):
        # Determine which segments to insert in this step
        start_idx = step * segments_per_step
        end_idx = min(start_idx + segments_per_step, total_segments)
        
        for idx in range(start_idx, end_idx):
            seg_id = segment_ids_ordered[idx]
            inserted_segments.add(seg_id)
            # Insert pixels from original image
            mask = segments == seg_id
            modified_image[mask] = image_np[mask]
        
        # Get model confidence after insertion
        pil_img = Image.fromarray(modified_image.astype('uint8'))
        tensor = transform(pil_img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(tensor)
            probs = torch.softmax(output, dim=1)
            confidence = probs[0, predicted_class].item()
        
        insertion_scores.append(confidence)
        
        if end_idx >= total_segments:
            break
    
    # Calculate AUC (higher is better for insertion)
    x = np.linspace(0, 1, len(insertion_scores))
    insertion_auc = np.trapz(insertion_scores, x)
    
    return insertion_scores, insertion_auc

In [ ]:
# ==============================================================================
# SECTION 7: RANDOM BASELINE COMPARISON
# ==============================================================================

def random_deletion_test(image_pil, segments, predicted_class, num_steps=20, 
                         replacement_value=0, num_runs=5):
    """
    Perform random-order deletion test as baseline comparison.
    
    Args:
        image_pil: Original PIL Image
        segments: Superpixel segmentation map
        predicted_class: Target class to track confidence for
        num_steps: Number of deletion steps
        replacement_value: Value to replace deleted pixels
        num_runs: Number of random runs to average
    
    Returns:
        avg_scores: Average confidence scores across runs
        avg_auc: Average AUC across runs
    """
    all_scores = []
    all_aucs = []
    
    unique_segments = np.unique(segments)
    total_segments = len(unique_segments)
    
    for run in range(num_runs):
        # Random shuffle of segments
        random_segments = list(unique_segments)
        random.shuffle(random_segments)
        
        # Create dummy importance scores (all equal)
        dummy_importance = {seg: 1 for seg in random_segments}
        
        # Reorder by random order
        ordered_importance = {seg: i for i, seg in enumerate(random_segments)}
        
        scores, auc = deletion_test(
            image_pil, segments, ordered_importance, predicted_class,
            num_steps=num_steps, replacement_value=replacement_value
        )
        
        all_scores.append(scores)
        all_aucs.append(auc)
    
    # Average across runs
    avg_scores = np.mean(all_scores, axis=0).tolist()
    avg_auc = np.mean(all_aucs)
    
    return avg_scores, avg_auc


def random_insertion_test(image_pil, segments, predicted_class, num_steps=20,
                          replacement_value=128, num_runs=5):
    """
    Perform random-order insertion test as baseline comparison.
    
    Args:
        image_pil: Original PIL Image
        segments: Superpixel segmentation map
        predicted_class: Target class to track confidence for
        num_steps: Number of insertion steps
        replacement_value: Value for non-inserted regions
        num_runs: Number of random runs to average
    
    Returns:
        avg_scores: Average confidence scores across runs
        avg_auc: Average AUC across runs
    """
    all_scores = []
    all_aucs = []
    
    unique_segments = np.unique(segments)
    total_segments = len(unique_segments)
    
    for run in range(num_runs):
        # Random shuffle of segments
        random_segments = list(unique_segments)
        random.shuffle(random_segments)
        
        # Create ordered importance based on random order
        ordered_importance = {seg: i for i, seg in enumerate(random_segments)}
        
        scores, auc = insertion_test(
            image_pil, segments, ordered_importance, predicted_class,
            num_steps=num_steps, replacement_value=replacement_value
        )
        
        all_scores.append(scores)
        all_aucs.append(auc)
    
    # Average across runs
    avg_scores = np.mean(all_scores, axis=0).tolist()
    avg_auc = np.mean(all_aucs)
    
    return avg_scores, avg_auc

In [ ]:
# ==============================================================================
# SECTION 8: COMPLETE FIDELITY EVALUATION PIPELINE
# ==============================================================================

def evaluate_fidelity(image_path, num_samples=1000, num_steps=20, num_random_runs=5):
    """
    Complete fidelity evaluation pipeline for a single image.
    
    Args:
        image_path: Path to the retinal fundus image
        num_samples: Number of LIME samples
        num_steps: Number of deletion/insertion steps
        num_random_runs: Number of random baseline runs
    
    Returns:
        Dictionary containing all fidelity metrics
    """
    print(f"\n{'='*60}")
    print(f"FIDELITY EVALUATION: {os.path.basename(image_path)}")
    print(f"{'='*60}")
    
    # Load image
    image_pil, image_tensor, _ = preprocess_image(image_path)
    
    # Get initial prediction
    pred_class, pred_conf, all_probs = get_model_prediction(image_tensor)
    pred_class_name = class_names[pred_class]
    print(f"\n✓ Model Prediction: {pred_class_name} (Class {pred_class})")
    print(f"  Confidence: {pred_conf:.4f}")
    
    # Generate LIME explanation
    print(f"\n→ Generating LIME explanation...")
    segments, importance_scores, lime_pred_class = generate_lime_explanation(
        image_pil, num_samples=num_samples
    )
    print(f"✓ LIME explanation generated")
    print(f"  Number of superpixels: {len(np.unique(segments))}")
    print(f"  Top 5 important segments: {sorted(importance_scores.items(), key=lambda x: abs(x[1]), reverse=True)[:5]}")
    
    # --- DELETION TEST ---
    print(f"\n→ Running Deletion Test (LIME-ordered)...")
    lime_deletion_scores, lime_deletion_auc = deletion_test(
        image_pil, segments, importance_scores, pred_class, num_steps=num_steps
    )
    print(f"✓ LIME Deletion AUC: {lime_deletion_auc:.4f}")
    
    print(f"\n→ Running Deletion Test (Random baseline)...")
    random_deletion_scores, random_deletion_auc = random_deletion_test(
        image_pil, segments, pred_class, num_steps=num_steps, num_runs=num_random_runs
    )
    print(f"✓ Random Deletion AUC: {random_deletion_auc:.4f}")
    
    # --- INSERTION TEST ---
    print(f"\n→ Running Insertion Test (LIME-ordered)...")
    lime_insertion_scores, lime_insertion_auc = insertion_test(
        image_pil, segments, importance_scores, pred_class, num_steps=num_steps
    )
    print(f"✓ LIME Insertion AUC: {lime_insertion_auc:.4f}")
    
    print(f"\n→ Running Insertion Test (Random baseline)...")
    random_insertion_scores, random_insertion_auc = random_insertion_test(
        image_pil, segments, pred_class, num_steps=num_steps, num_runs=num_random_runs
    )
    print(f"✓ Random Insertion AUC: {random_insertion_auc:.4f}")
    
    # --- CALCULATE METRICS ---
    deletion_improvement = random_deletion_auc / lime_deletion_auc if lime_deletion_auc > 0 else 0
    insertion_improvement = lime_insertion_auc / random_insertion_auc if random_insertion_auc > 0 else 0
    
    print(f"\n{'='*60}")
    print("FIDELITY METRICS SUMMARY")
    print(f"{'='*60}")
    print(f"Deletion Test:")
    print(f"  LIME AUC:    {lime_deletion_auc:.4f}")
    print(f"  Random AUC:  {random_deletion_auc:.4f}")
    print(f"  Improvement: {deletion_improvement:.2f}x (lower AUC is better)")
    print(f"\nInsertion Test:")
    print(f"  LIME AUC:    {lime_insertion_auc:.4f}")
    print(f"  Random AUC:  {random_insertion_auc:.4f}")
    print(f"  Improvement: {insertion_improvement:.2f}x (higher AUC is better)")
    
    return {
        'image_path': image_path,
        'predicted_class': pred_class_name,
        'predicted_confidence': pred_conf,
        'lime_deletion_auc': lime_deletion_auc,
        'random_deletion_auc': random_deletion_auc,
        'deletion_improvement': deletion_improvement,
        'lime_insertion_auc': lime_insertion_auc,
        'random_insertion_auc': random_insertion_auc,
        'insertion_improvement': insertion_improvement,
        'lime_deletion_scores': lime_deletion_scores,
        'random_deletion_scores': random_deletion_scores,
        'lime_insertion_scores': lime_insertion_scores,
        'random_insertion_scores': random_insertion_scores,
        'segments': segments,
        'importance_scores': importance_scores
    }

In [ ]:
# ==============================================================================
# SECTION 9: VISUALIZATION FUNCTIONS
# ==============================================================================

def plot_fidelity_curves(results, save_path=None):
    """
    Plot Deletion and Insertion curves comparing LIME vs Random baseline.
    
    Args:
        results: Dictionary from evaluate_fidelity()
        save_path: Path to save the figure (optional)
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # --- Deletion Curve ---
    ax1 = axes[0]
    steps = np.linspace(0, 100, len(results['lime_deletion_scores']))
    
    ax1.plot(steps, results['lime_deletion_scores'], 'b-', linewidth=2.5, 
             label=f"LIME (AUC={results['lime_deletion_auc']:.3f})")
    ax1.plot(steps, results['random_deletion_scores'], 'r--', linewidth=2.5,
             label=f"Random (AUC={results['random_deletion_auc']:.3f})")
    
    ax1.fill_between(steps, results['lime_deletion_scores'], alpha=0.2, color='blue')
    ax1.fill_between(steps, results['random_deletion_scores'], alpha=0.2, color='red')
    
    ax1.set_xlabel('Percentage of Pixels Removed (%)', fontsize=12)
    ax1.set_ylabel('Model Confidence for Predicted Class', fontsize=12)
    ax1.set_title(f'Deletion Test\n{results["predicted_class"]} - {results["deletion_improvement"]:.1f}x Improvement', fontsize=14)
    ax1.legend(loc='upper right', fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, 100)
    ax1.set_ylim(0, 1.05)
    
    # --- Insertion Curve ---
    ax2 = axes[1]
    steps = np.linspace(0, 100, len(results['lime_insertion_scores']))
    
    ax2.plot(steps, results['lime_insertion_scores'], 'g-', linewidth=2.5,
             label=f"LIME (AUC={results['lime_insertion_auc']:.3f})")
    ax2.plot(steps, results['random_insertion_scores'], 'r--', linewidth=2.5,
             label=f"Random (AUC={results['random_insertion_auc']:.3f})")
    
    ax2.fill_between(steps, results['lime_insertion_scores'], alpha=0.2, color='green')
    ax2.fill_between(steps, results['random_insertion_scores'], alpha=0.2, color='red')
    
    ax2.set_xlabel('Percentage of Pixels Inserted (%)', fontsize=12)
    ax2.set_ylabel('Model Confidence for Predicted Class', fontsize=12)
    ax2.set_title(f'Insertion Test\n{results["predicted_class"]} - {results["insertion_improvement"]:.1f}x Improvement', fontsize=14)
    ax2.legend(loc='lower right', fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0, 100)
    ax2.set_ylim(0, 1.05)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ Figure saved to: {save_path}")
    
    plt.show()


def plot_comparison_bar(all_results, save_path=None):
    """
    Plot bar chart comparing LIME vs Random AUC across multiple images.
    
    Args:
        all_results: List of result dictionaries
        save_path: Path to save the figure (optional)
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Prepare data
    image_names = [os.path.basename(r['image_path'])[:15] + '...' for r in all_results]
    lime_del = [r['lime_deletion_auc'] for r in all_results]
    rand_del = [r['random_deletion_auc'] for r in all_results]
    lime_ins = [r['lime_insertion_auc'] for r in all_results]
    rand_ins = [r['random_insertion_auc'] for r in all_results]
    
    x = np.arange(len(image_names))
    width = 0.35
    
    # Deletion comparison
    ax1 = axes[0]
    bars1 = ax1.bar(x - width/2, lime_del, width, label='LIME', color='steelblue')
    bars2 = ax1.bar(x + width/2, rand_del, width, label='Random', color='lightcoral')
    
    ax1.set_ylabel('Deletion AUC (Lower = Better)', fontsize=11)
    ax1.set_title('Deletion Test Comparison', fontsize=13)
    ax1.set_xticks(x)
    ax1.set_xticklabels(image_names, rotation=45, ha='right', fontsize=9)
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Insertion comparison
    ax2 = axes[1]
    bars3 = ax2.bar(x - width/2, lime_ins, width, label='LIME', color='seagreen')
    bars4 = ax2.bar(x + width/2, rand_ins, width, label='Random', color='lightcoral')
    
    ax2.set_ylabel('Insertion AUC (Higher = Better)', fontsize=11)
    ax2.set_title('Insertion Test Comparison', fontsize=13)
    ax2.set_xticks(x)
    ax2.set_xticklabels(image_names, rotation=45, ha='right', fontsize=9)
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ Figure saved to: {save_path}")
    
    plt.show()

In [ ]:
# ==============================================================================
# SECTION 10: RUN FIDELITY EVALUATION ON SAMPLE IMAGES
# ==============================================================================

# Define test images (modify paths as needed)
test_images = [
    '../data/colored_images/Severe/0c917c5c5e1c.png',  # Severe DR case
    '../data/colored_images/No_DR/0c917c5c5e1c.png',   # No DR case
    '../data/colored_images/Moderate/002c21358ce6.png' # Moderate case
]

# Filter to existing files
existing_images = [img for img in test_images if os.path.exists(img)]

if len(existing_images) == 0:
    print("No test images found. Please update the paths above.")
    print("\nAvailable directories:")
    data_dir = '../data/colored_images'
    if os.path.exists(data_dir):
        for class_dir in os.listdir(data_dir):
            class_path = os.path.join(data_dir, class_dir)
            if os.path.isdir(class_path):
                files = os.listdir(class_path)[:3]
                print(f"  {class_dir}: {files}")
else:
    print(f"Found {len(existing_images)} test images")

In [ ]:
# ==============================================================================
# SECTION 11: EVALUATE MULTIPLE IMAGES AND COMPILE RESULTS
# ==============================================================================

all_results = []

for img_path in existing_images:
    try:
        result = evaluate_fidelity(
            img_path,
            num_samples=1000,
            num_steps=20,
            num_random_runs=5
        )
        all_results.append(result)
        
        # Plot individual curves
        plot_fidelity_curves(result, save_path=f"fidelity_curves_{os.path.basename(img_path)}.png")
        
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        continue

In [ ]:
# ==============================================================================
# SECTION 12: AGGREGATE RESULTS AND SUMMARY STATISTICS
# ==============================================================================

if len(all_results) > 0:
    print(f"\n{'='*70}")
    print("AGGREGATE FIDELITY METRICS SUMMARY")
    print(f"{'='*70}")
    print(f"Number of images evaluated: {len(all_results)}")
    print(f"\n{'='*70}")
    
    # Calculate averages
    avg_lime_del = np.mean([r['lime_deletion_auc'] for r in all_results])
    avg_rand_del = np.mean([r['random_deletion_auc'] for r in all_results])
    avg_lime_ins = np.mean([r['lime_insertion_auc'] for r in all_results])
    avg_rand_ins = np.mean([r['random_insertion_auc'] for r in all_results])
    avg_del_improvement = avg_rand_del / avg_lime_del if avg_lime_del > 0 else 0
    avg_ins_improvement = avg_lime_ins / avg_rand_ins if avg_rand_ins > 0 else 0
    
    print("DELETION TEST RESULTS")
    print(f"-" * 40)
    print(f"  Average LIME Deletion AUC:   {avg_lime_del:.4f}")
    print(f"  Average Random Deletion AUC: {avg_rand_del:.4f}")
    print(f"  Average Improvement:          {avg_del_improvement:.2f}x")
    print(f"\nINSERTION TEST RESULTS")
    print(f"-" * 40)
    print(f"  Average LIME Insertion AUC:   {avg_lime_ins:.4f}")
    print(f"  Average Random Insertion AUC: {avg_rand_ins:.4f}")
    print(f"  Average Improvement:          {avg_ins_improvement:.2f}x")
    print(f"\n{'='*70}")
    
    # Create comparison bar chart
    plot_comparison_bar(all_results, save_path='fidelity_comparison_bars.png')
    
    # Save results to JSON
    import json
    results_json = {
        'summary': {
            'num_images': len(all_results),
            'avg_lime_deletion_auc': avg_lime_del,
            'avg_random_deletion_auc': avg_rand_del,
            'deletion_improvement_factor': avg_del_improvement,
            'avg_lime_insertion_auc': avg_lime_ins,
            'avg_random_insertion_auc': avg_rand_ins,
            'insertion_improvement_factor': avg_ins_improvement
        },
        'individual_results': [{k: v for k, v in r.items() if k not in ['segments', 'importance_scores', 
                                                                         'lime_deletion_scores', 'random_deletion_scores',
                                                                         'lime_insertion_scores', 'random_insertion_scores']} 
                              for r in all_results]
    }
    
    with open('fidelity_metrics_results.json', 'w') as f:
        json.dump(results_json, f, indent=2)
    print(f"\n✓ Results saved to: fidelity_metrics_results.json")
else:
    print("No results to summarize. Please run the evaluation on test images first.")

In [ ]:
# ==============================================================================
# SECTION 13: GENERATE TABLE FOR MANUSCRIPT
# ==============================================================================

if len(all_results) > 0:
    print("\n" + "="*70)
    print("TABLE FOR MANUSCRIPT")
    print("="*70)
    print("\nTable: Fidelity Metric Results - Deletion and Insertion AUC Scores")
    print("-"*70)
    print(f"{'Image':<20} {'Class':<12} {'Del AUC':<10} {'Rand Del':<10} {'Ins AUC':<10} {'Rand Ins':<10}")
    print("-"*70)
    for r in all_results:
        img_name = os.path.basename(r['image_path'])[:18]
        print(f"{img_name:<20} {r['predicted_class']:<12} "
              f"{r['lime_deletion_auc']:<10.4f} {r['random_deletion_auc']:<10.4f} "
              f"{r['lime_insertion_auc']:<10.4f} {r['random_insertion_auc']:<10.4f}")
    print("-"*70)
    print(f"{'AVERAGE':<20} {'':<12} {avg_lime_del:<10.4f} {avg_rand_del:<10.4f} "
          f"{avg_lime_ins:<10.4f} {avg_rand_ins:<10.4f}")
    print("="*70)

## Interpretation of Results

### Deletion Test
- **Lower AUC = More Faithful Explanation**
- If LIME correctly identifies important regions, removing them should cause rapid confidence drop
- A significant gap between LIME AUC and Random AUC indicates faithful explanations

### Insertion Test
- **Higher AUC = More Faithful Explanation**
- Starting from a blank image, adding important regions should rapidly restore confidence
- A significant gap between LIME AUC and Random AUC indicates the identified regions are sufficient for prediction

### Faithfulness Ratio
- Deletion Improvement Factor = Random AUC / LIME AUC (higher = more faithful)
- Insertion Improvement Factor = LIME AUC / Random AUC (higher = more faithful)

### Clinical Significance
These metrics provide mathematical proof that the LIME-identified superpixels correspond to clinically relevant features (hard exudates, microaneurysms) rather than image artifacts or background noise. This validation is critical for building clinical trust in the AI system.